# From Hackathon AI to Production AI

## Chapter 02 — Will It Survive Reality?

Version 2.0

## Story So Far


Tuesday morning.

HelpDeskAI has now been deployed to a pilot group.

Everything looked perfect yesterday.

Today, reality arrives.

One engineer reports that the application stopped responding.

Another receives a timeout.

A third user submits an empty ticket.

Someone pastes an entire email conversation containing
thousands of words.

The Gemini model is functioning correctly.

The software around it is not prepared for failure.

This is the moment every prototype becomes an engineering
project.

## Workshop Context


In Chapter 00 we proved that Gemini could solve our problem.

In Chapter 01 we learned how to observe our application.

Today we focus on resilience.

Production AI systems must continue operating even when
unexpected situations occur.

Instead of assuming success, engineers prepare for failure.

## Learning Outcomes

- Validate user input
- Handle runtime errors
- Implement retry logic
- Design graceful failure
- Record failed requests
- Measure reliability

## Today's Engineering Question



How do we build AI applications that continue working

when things go wrong?

## Engineering Principle



Reliable systems prepare for failure

instead of assuming success.

## Looking Back


Yesterday we focused on observability.

We learned how to answer questions such as

• What prompt was used?

• How long did it take?

• What did the model return?

Today we ask a different question.

What happens when something fails?

Every production system eventually encounters unexpected
inputs, network issues, temporary outages, or human mistakes.

Engineering begins when we plan for those situations.

## Engineering Discussion


Many software failures are not caused by the AI model.

Instead they originate from the surrounding application.

Examples include

• empty user input

• invalid configuration

• temporary network failures

• API rate limits

• interrupted internet connections

• programming mistakes

A reliable AI application expects these situations and
responds gracefully instead of crashing.

## Today's Lab


Throughout this notebook we will transform HelpDeskAI from
an observable prototype into a resilient application.

We will improve the software around the model rather than
the model itself.

The goal is not perfection.

The goal is recovery.

## Hands-on Exercise 1

### Import Required Libraries


Today's experiments introduce error handling and retry logic.

Import the additional modules needed for reliability.

In [ ]:
import time
from datetime import datetime

from google import genai

## Hands-on Exercise 2

### Configure Gemini Client


Reuse the API key from the previous chapter.

In [ ]:
try:
    from google.colab import userdata

    API_KEY = userdata.get("GOOGLE_API_KEY")

except Exception:

    API_KEY = input(
        "Enter your Gemini API Key: "
    ).strip()

client = genai.Client(
    api_key=API_KEY
)

print("Client ready.")

## Reflection


Every production incident begins with an assumption that
turned out to be false.

Examples include

"I assumed the user would enter text."

"I assumed the API would always respond."

"I assumed the internet connection would never fail."

Reliable software challenges assumptions before users do.

## Hands-on Exercise 3

### Validate User Input


Before contacting Gemini, verify that the ticket contains
useful information.

Never send obviously invalid requests to the API.

In [ ]:
def validate_ticket(ticket):

    if ticket is None:
        return False

    if not isinstance(ticket, str):
        return False

    if len(ticket.strip()) == 0:
        return False

    return True

## Hands-on Exercise 4

### Test Input Validation


Engineers test edge cases before users discover them.

In [ ]:
samples = [

    "",

    "   ",

    None,

    42,

    "My account has been locked."

]

for sample in samples:

    print(sample, "->", validate_ticket(sample))

## Engineering Discussion


Notice that we have not contacted Gemini yet.

Good engineering often means rejecting invalid requests
before they consume time, money, or API quota.

Validation is one of the simplest and most effective forms
of reliability.

## Reflection


Input validation prevents avoidable failures.

Unfortunately, not every failure can be prevented.

Networks become unavailable.

Servers become busy.

Requests time out.

Good software does not panic when these situations occur.

Instead, it responds predictably and recovers whenever
possible.

## Hands-on Exercise 5

### Handling Exceptions


Python exceptions allow our application to recover from
unexpected situations instead of terminating.

Let's begin by wrapping the API call inside a try/except
block.

In [ ]:
def classify_once(ticket):

    prompt = f'''
You are a customer support classifier.

Return only one category.

Billing
Technical
Account

Ticket

{ticket}
'''

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text.strip()

## Hands-on Exercise 6

### Build a Safe Classifier


Catch runtime errors and return a friendly message instead
of crashing the notebook.

In [ ]:
def safe_classify(ticket):

    if not validate_ticket(ticket):

        return {
            "success": False,
            "error": "Invalid ticket."
        }

    try:

        prediction = classify_once(ticket)

        return {

            "success": True,

            "prediction": prediction

        }

    except Exception as error:

        return {

            "success": False,

            "error": str(error)

        }

## Hands-on Exercise 7

### Test Graceful Failure


Notice that invalid input is handled without generating an
exception.

In [ ]:
tests = [

    "",

    None,

    "The application crashes after login."

]

for ticket in tests:

    print(safe_classify(ticket))

## Engineering Discussion


Graceful failure is an important engineering principle.

A production application should not expose stack traces to
users.

Instead, it should recover whenever possible and provide
clear, actionable feedback.

## Reflection


Many cloud failures are temporary.

If a request fails once, trying again a few seconds later
often succeeds.

This is why retries are common in distributed systems.

## Hands-on Exercise 8

### Retry Failed Requests


Implement a simple retry mechanism.

Instead of giving up immediately, attempt the request
multiple times before reporting failure.

In [ ]:
def classify_with_retry(ticket, retries=3):

    for attempt in range(retries):

        try:

            prediction = classify_once(ticket)

            return {

                "success": True,

                "prediction": prediction,

                "attempts": attempt + 1

            }

        except Exception as error:

            print(
                f"Attempt {attempt + 1} failed."
            )

            if attempt < retries - 1:

                time.sleep(2)

            else:

                return {

                    "success": False,

                    "error": str(error),

                    "attempts": retries

                }

## Hands-on Exercise 9

### Test Retry Logic


Execute the retry function using a valid support ticket.

In [ ]:
result = classify_with_retry(

    "My monthly invoice contains duplicate charges."

)

print(result)

## Think Like an Engineer


A retry is not about fixing the AI model.

It is about acknowledging that distributed systems are not
perfect.

Reliable applications are patient.

They expect temporary failures and recover automatically
whenever it is safe to do so.

## Hands-on Exercise 10

### Log Failed Requests


Production systems record failures for later investigation.

Let's build a simple failure log.

In [ ]:
failed_requests = []

def log_failure(ticket, error):

    failed_requests.append({

        "timestamp": datetime.now().isoformat(),

        "ticket": ticket,

        "error": str(error)

    })

## Hands-on Exercise 11

### Integrate Failure Logging


Update the retry function so that permanently failed
requests are recorded automatically.

In [ ]:
# Inside classify_with_retry(), before returning the final
# failure result, call:

# log_failure(ticket, error)

# Then inspect the log.

print(f"Failures recorded: {len(failed_requests)}")

for failure in failed_requests:

    print(failure)

## Reflection


Input validation prevents avoidable failures.

Unfortunately, not every failure can be prevented.

Networks become unavailable.

Servers become busy.

Requests time out.

Good software does not panic when these situations occur.

Instead, it responds predictably and recovers whenever
possible.

## Hands-on Exercise 5

### Handling Exceptions


Python exceptions allow our application to recover from
unexpected situations instead of terminating.

Let's begin by wrapping the API call inside a try/except
block.

In [ ]:
def classify_once(ticket):

    prompt = f'''
You are a customer support classifier.

Return only one category.

Billing
Technical
Account

Ticket

{ticket}
'''

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text.strip()

## Hands-on Exercise 6

### Build a Safe Classifier


Catch runtime errors and return a friendly message instead
of crashing the notebook.

In [ ]:
def safe_classify(ticket):

    if not validate_ticket(ticket):

        return {
            "success": False,
            "error": "Invalid ticket."
        }

    try:

        prediction = classify_once(ticket)

        return {

            "success": True,

            "prediction": prediction

        }

    except Exception as error:

        return {

            "success": False,

            "error": str(error)

        }

## Hands-on Exercise 7

### Test Graceful Failure


Notice that invalid input is handled without generating an
exception.

In [ ]:
tests = [

    "",

    None,

    "The application crashes after login."

]

for ticket in tests:

    print(safe_classify(ticket))

## Engineering Discussion


Graceful failure is an important engineering principle.

A production application should not expose stack traces to
users.

Instead, it should recover whenever possible and provide
clear, actionable feedback.

## Reflection


Many cloud failures are temporary.

If a request fails once, trying again a few seconds later
often succeeds.

This is why retries are common in distributed systems.

## Hands-on Exercise 8

### Retry Failed Requests


Implement a simple retry mechanism.

Instead of giving up immediately, attempt the request
multiple times before reporting failure.

In [ ]:
def classify_with_retry(ticket, retries=3):

    for attempt in range(retries):

        try:

            prediction = classify_once(ticket)

            return {

                "success": True,

                "prediction": prediction,

                "attempts": attempt + 1

            }

        except Exception as error:

            print(
                f"Attempt {attempt + 1} failed."
            )

            if attempt < retries - 1:

                time.sleep(2)

            else:

                return {

                    "success": False,

                    "error": str(error),

                    "attempts": retries

                }

## Hands-on Exercise 9

### Test Retry Logic


Execute the retry function using a valid support ticket.

In [ ]:
result = classify_with_retry(

    "My monthly invoice contains duplicate charges."

)

print(result)

## Think Like an Engineer


A retry is not about fixing the AI model.

It is about acknowledging that distributed systems are not
perfect.

Reliable applications are patient.

They expect temporary failures and recover automatically
whenever it is safe to do so.

## Hands-on Exercise 10

### Log Failed Requests


Production systems record failures for later investigation.

Let's build a simple failure log.

In [ ]:
failed_requests = []

def log_failure(ticket, error):

    failed_requests.append({

        "timestamp": datetime.now().isoformat(),

        "ticket": ticket,

        "error": str(error)

    })

## Hands-on Exercise 11

### Integrate Failure Logging


Update the retry function so that permanently failed
requests are recorded automatically.

In [ ]:
# Inside classify_with_retry(), before returning the final
# failure result, call:

# log_failure(ticket, error)

# Then inspect the log.

print(f"Failures recorded: {len(failed_requests)}")

for failure in failed_requests:

    print(failure)

## Reflection


Resilience is not about eliminating failures.

It is about reducing the impact of failures.

Users expect software to behave predictably, even when
unexpected events occur.

A resilient application degrades gracefully instead of
stopping completely.

## Hands-on Exercise 12

### Measure Reliability


Engineers use metrics to understand how reliable their
systems are.

Let's compute a few simple reliability statistics.

In [ ]:
successful = 0
failed = 0

results = [

    safe_classify("Reset my password."),

    safe_classify(""),

    safe_classify("My invoice is incorrect."),

    safe_classify(None),

    safe_classify("The mobile app crashes during login.")

]

for result in results:

    if result["success"]:
        successful += 1
    else:
        failed += 1

print(f"Successful Requests : {successful}")
print(f"Failed Requests     : {failed}")

total = successful + failed

if total > 0:

    print(f"Success Rate        : {successful / total:.1%}")
    print(f"Failure Rate        : {failed / total:.1%}")

## Hands-on Exercise 13

### Review the Failure Log


Failures should be opportunities to learn.

Inspect the information collected during unsuccessful
requests.

In [ ]:
if failed_requests:

    for failure in failed_requests:

        print("-" * 60)

        print(f"Time   : {failure['timestamp']}")
        print(f"Ticket : {failure['ticket']}")
        print(f"Error  : {failure['error']}")

else:

    print("No failures have been recorded.")

## Engineering Discussion


Reliability is measured over time rather than by individual
requests.

One successful API call proves very little.

Thousands of successful requests, combined with consistent
recovery from temporary failures, demonstrate that a system
is dependable.

Production engineers look for trends, not isolated events.

## Hands-on Exercise 14

### Design a Resilient Workflow


Review the lifecycle of a request.

Where should validation, retries, logging, and error
handling occur?

Sketch your workflow before looking at the example below.

In [ ]:
Request

   │

   ▼

Validate Input

   │

   ▼

Call Gemini

   │

   ├──────── Success ─────────► Return Prediction

   │

   ▼

Exception

   │

   ▼

Retry

   │

   ├──────── Success ─────────► Return Prediction

   │

   ▼

Log Failure

   │

   ▼

Return Friendly Error Message

## Challenge Exercise


Build a production-style classify_ticket() function that

• validates user input

• retries failed requests

• records failed requests

• measures latency

• returns structured JSON

• never crashes the application

Compare your solution with the functions developed
throughout this chapter.

## Think Like an Engineer


A prototype demonstrates possibility.

A production system demonstrates reliability.

Professional engineers assume that every dependency can
fail.

They design software that continues serving users despite
those failures.

Reliability is not a feature that can be added later.

It is an architectural decision made from the beginning.

## Checkpoint


Your HelpDeskAI application can now

• validate incoming requests

• reject invalid input

• recover from runtime exceptions

• retry transient failures

• record failed requests

• compute basic reliability metrics

The prototype has evolved into a resilient application.

## Chapter Summary


This chapter introduced one of the most important concepts
in production AI engineering: resilience.

Rather than assuming every API call would succeed, we
designed software that validates input, handles exceptions,
retries temporary failures, and records problems for later
analysis.

The Gemini model did not change.

Our engineering practices did.

Reliable AI systems are built by preparing for failure,
measuring reliability, and continuously improving recovery
strategies.

## Production Readiness Checklist

Completed

☑ Input validation
☑ Graceful error handling
☑ Retry logic
☑ Failure logging
☑ Reliability metrics
☑ Structured responses
☑ Basic resilience

Still Missing

☐ Output evaluation
☐ Hallucination detection
☐ Grounding
☐ Prompt testing
☐ Automated evaluation
☐ Monitoring dashboard
☐ Security review
☐ Deployment

## Looking Ahead

### Chapter 03 — Can We Trust It?


Our application is now reliable.

But another question remains.

What if the model confidently produces an incorrect answer?

Reliable software can still generate unreliable outputs.

In the next chapter we explore evaluation, validation,
grounding, and the engineering techniques used to build
trustworthy AI systems.

## Reflection


Chapter 00 asked

Can it answer?

Chapter 01 asked

Do we understand it?

Chapter 02 asked

Will it survive reality?

A production AI system must answer all three questions
before it is ready for users.